# 📊 Task 01 — Dataset Overview
### Disease Surveillance Project — Full Star Schema (All 10 Files)

**Fact tables (5):** disease surveillance, outbreak, environmental, health programs, lab & healthcare
**Dimension tables (5):** dates, state, disease, program, source

**Objective:** Load and profile all 10 files, confirm how the fact tables link to the dimension tables, and produce a data dictionary for the whole dataset that the rest of the team (Data Quality, Disease Analysis, etc.) can build on.

---
### ⚠️ INPUT NEEDED FROM YOU
1. **Cell 2** — set `ZIP_PATH` to the path of your uploaded zip file (e.g. `/content/EDA Dataset.zip` in Colab). It will be unzipped automatically and `DATA_FOLDER` will be set for you.

That's it — everything else runs automatically using the real column names from your files.


In [ ]:
# Cell 1: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import zipfile
import glob
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Libraries imported successfully.")


In [ ]:
# Cell 2: ===== INPUT REQUIRED =====
ZIP_PATH = "/content/EDA Dataset.zip"   # <-- CHANGE THIS if your zip file has a different path/name
EXTRACT_TO = "/content/EDA_Dataset_Extracted"   # folder where the zip contents will be extracted

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_TO)
    print(f"Extracted '{ZIP_PATH}' to '{EXTRACT_TO}'")
else:
    raise FileNotFoundError(f"Zip file not found at: {ZIP_PATH}. Update ZIP_PATH above.")

# The zip may extract directly, or into a subfolder (e.g. 'EDA Dataset/') — find wherever the CSVs actually landed
csv_paths = glob.glob(os.path.join(EXTRACT_TO, '**', '*.csv'), recursive=True)
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found after extracting to {EXTRACT_TO}")

DATA_FOLDER = os.path.dirname(csv_paths[0])
print(f"DATA_FOLDER set to: '{DATA_FOLDER}'")
print(f"Found {len(csv_paths)} CSV files:")
for p in csv_paths:
    print(f"  - {os.path.basename(p)}")


In [ ]:
# Cell 3: Load all 10 files into a dictionary
FILES = {
    "fact_disease_surveillance": "fact_disease_surveillance_cleaned.csv",
    "fact_outbreak":             "fact outbreak cleaned.csv",          # note: space, not underscore
    "fact_environmental":        "fact_environmental_cleaned.csv",
    "fact_health_programs":      "fact_health_programs_cleaned.csv",
    "fact_lab_healthcare":       "fact_lab_healthcare_cleaned.csv",
    "dim_dates":                 "dim_dates_cleaned.csv",
    "dim_state":                 "dim_state_cleaned.csv",
    "dim_disease":               "dim_disease_cleaned.csv",
    "dim_program":                "dim_program_cleaned.csv",
    "dim_source":                "dim_source_cleaned.csv",
}

dfs = {}
for key, filename in FILES.items():
    path = os.path.join(DATA_FOLDER, filename)
    dfs[key] = pd.read_csv(path)
    print(f"{key:28s} loaded  -> shape {dfs[key].shape}")

# Convenience references
fact_disease = dfs["fact_disease_surveillance"]
fact_outbreak = dfs["fact_outbreak"]
fact_environmental = dfs["fact_environmental"]
fact_health_programs = dfs["fact_health_programs"]
fact_lab_healthcare = dfs["fact_lab_healthcare"]
dim_dates = dfs["dim_dates"]
dim_state = dfs["dim_state"]
dim_disease = dfs["dim_disease"]
dim_program = dfs["dim_program"]
dim_source = dfs["dim_source"]


## 1. Star Schema Structure

In [ ]:
# Cell 3: Visualize the schema relationships
print("STAR SCHEMA STRUCTURE")
print("="*70)
print("FACT TABLES (5)                          DIMENSION TABLES (5)")
print("-"*70)
print("fact_disease_surveillance  --date_id----> dim_dates")
print("                           --state_id---> dim_state")
print("                           --disease_id-> dim_disease")
print("                           --source_id--> dim_source")
print()
print("fact_outbreak              --date_id----> dim_dates")
print("                           --state_id---> dim_state")
print("                           --disease_id-> dim_disease")
print("                           --source_id--> dim_source")
print()
print("fact_environmental         --date_id----> dim_dates")
print("                           --state_id---> dim_state")
print()
print("fact_health_programs       --date_id----> dim_dates")
print("                           --state_id---> dim_state")
print("                           --program_id-> dim_program")
print()
print("fact_lab_healthcare        --date_id----> dim_dates")
print("                           --state_id---> dim_state")
print("="*70)


## 2. Per-File Shape Summary

In [ ]:
# Cell 4: Shape of every file in one table
shape_summary = pd.DataFrame({
    'File': list(dfs.keys()),
    'Rows': [d.shape[0] for d in dfs.values()],
    'Columns': [d.shape[1] for d in dfs.values()],
})
display(shape_summary)


## 3. Detailed Profile — Per File

The loop below runs a full profile (head, dtypes, describe, missing values, duplicates, unique counts) for **every one of the 10 files**, one section at a time.


In [ ]:
# Cell 5: Full profiling loop for all 10 files
def profile_file(name, df):
    print("\n" + "="*80)
    print(f"FILE: {name}   |   Shape: {df.shape}")
    print("="*80)

    print("\n-- Columns & Data Types --")
    print(df.dtypes)

    print("\n-- Head (first 3 rows) --")
    display(df.head(3))

    print("\n-- Numeric Summary --")
    num_summary = df.describe().T
    if not num_summary.empty:
        display(num_summary)
    else:
        print("No numeric columns.")

    print("\n-- Missing Values --")
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if not missing.empty:
        display(missing)
    else:
        print("No missing values. ✅")

    print(f"\n-- Duplicate Rows: {df.duplicated().sum()} --")

    print("\n-- Unique Values per Column --")
    display(df.nunique())

for name, df in dfs.items():
    profile_file(name, df)


## 4. Referential Integrity Check (Foreign Keys → Dimension Tables)

In [ ]:
# Cell 6: Check that every fact-table foreign key has a matching dimension record
def check_fk(fact_name, fact_df, fk_col, dim_name, dim_df, dim_key):
    if fk_col not in fact_df.columns:
        return
    fact_ids = set(fact_df[fk_col].unique())
    dim_ids = set(dim_df[dim_key].unique())
    orphans = fact_ids - dim_ids
    status = "OK" if not orphans else "ORPHANS FOUND"
    print(f"{fact_name:28s} [{fk_col:12s}] -> {dim_name:12s} | unique keys: {len(fact_ids):4d} | orphans: {len(orphans):3d}  [{status}]")
    if orphans:
        print(f"   Sample orphan keys: {list(orphans)[:10]}")

print("Referential Integrity Check — all fact tables vs their dimension tables:\n")

checks = [
    ("fact_disease_surveillance", fact_disease, "date_id", "dim_dates", dim_dates, "date_id"),
    ("fact_disease_surveillance", fact_disease, "state_id", "dim_state", dim_state, "state_id"),
    ("fact_disease_surveillance", fact_disease, "disease_id", "dim_disease", dim_disease, "disease_id"),
    ("fact_disease_surveillance", fact_disease, "source_id", "dim_source", dim_source, "source_id"),

    ("fact_outbreak", fact_outbreak, "date_id", "dim_dates", dim_dates, "date_id"),
    ("fact_outbreak", fact_outbreak, "state_id", "dim_state", dim_state, "state_id"),
    ("fact_outbreak", fact_outbreak, "disease_id", "dim_disease", dim_disease, "disease_id"),
    ("fact_outbreak", fact_outbreak, "source_id", "dim_source", dim_source, "source_id"),

    ("fact_environmental", fact_environmental, "date_id", "dim_dates", dim_dates, "date_id"),
    ("fact_environmental", fact_environmental, "state_id", "dim_state", dim_state, "state_id"),

    ("fact_health_programs", fact_health_programs, "date_id", "dim_dates", dim_dates, "date_id"),
    ("fact_health_programs", fact_health_programs, "state_id", "dim_state", dim_state, "state_id"),
    ("fact_health_programs", fact_health_programs, "program_id", "dim_program", dim_program, "program_id"),

    ("fact_lab_healthcare", fact_lab_healthcare, "date_id", "dim_dates", dim_dates, "date_id"),
    ("fact_lab_healthcare", fact_lab_healthcare, "state_id", "dim_state", dim_state, "state_id"),
]

for c in checks:
    check_fk(*c)


## 5. Time Range Coverage (per fact table)

In [ ]:
# Cell 7: Date range covered by each fact table
dim_dates_parsed = dim_dates.copy()
dim_dates_parsed['full_date'] = pd.to_datetime(dim_dates_parsed['full_date'])

for name, df in [("fact_disease_surveillance", fact_disease),
                  ("fact_outbreak", fact_outbreak),
                  ("fact_environmental", fact_environmental),
                  ("fact_health_programs", fact_health_programs),
                  ("fact_lab_healthcare", fact_lab_healthcare)]:
    merged = df.merge(dim_dates_parsed[['date_id', 'full_date']], on='date_id', how='left')
    print(f"{name:28s} : {merged['full_date'].min().date()}  to  {merged['full_date'].max().date()}  "
          f"({df['date_id'].nunique()} distinct dates)")


## 6. Geographic Scope (per fact table)

In [ ]:
# Cell 8: States covered by each fact table
for name, df in [("fact_disease_surveillance", fact_disease),
                  ("fact_outbreak", fact_outbreak),
                  ("fact_environmental", fact_environmental),
                  ("fact_health_programs", fact_health_programs),
                  ("fact_lab_healthcare", fact_lab_healthcare)]:
    print(f"{name:28s} : {df['state_id'].nunique()} unique states covered")


In [ ]:
# Cell 9: Full state list with region info
display(dim_state[['state_id', 'state_name', 'region', 'population']])


## 7. Disease & Program & Source Reference Lists

In [ ]:
# Cell 10: Reference dimension contents
print("Diseases:")
display(dim_disease)

print("\nHealth Programs:")
display(dim_program)

print("\nData Sources:")
display(dim_source)


## 8. Cross-File Consistency Check

In [ ]:
# Cell 11: Confirm state_id, date_id ranges are consistent across all fact tables
print("date_id range per fact table:")
for name, df in [("fact_disease_surveillance", fact_disease),
                  ("fact_outbreak", fact_outbreak),
                  ("fact_environmental", fact_environmental),
                  ("fact_health_programs", fact_health_programs),
                  ("fact_lab_healthcare", fact_lab_healthcare)]:
    print(f"  {name:28s} : min={df['date_id'].min()}, max={df['date_id'].max()}")

print("\nstate_id range per fact table:")
for name, df in [("fact_disease_surveillance", fact_disease),
                  ("fact_outbreak", fact_outbreak),
                  ("fact_environmental", fact_environmental),
                  ("fact_health_programs", fact_health_programs),
                  ("fact_lab_healthcare", fact_lab_healthcare)]:
    print(f"  {name:28s} : min={df['state_id'].min()}, max={df['state_id'].max()}")


## 9. Master Data Dictionary Export (All 10 Files)

In [ ]:
# Cell 12: Auto-generate one combined data dictionary covering every file
all_dicts = []
for name, df in dfs.items():
    d = pd.DataFrame({
        'File': name,
        'Column Name': df.columns,
        'Data Type': df.dtypes.values,
        'Non-Null Count': df.notnull().sum().values,
        'Missing %': (df.isnull().sum() / len(df) * 100).round(2).values,
        'Unique Values': df.nunique().values,
        'Sample Value': [df[col].dropna().iloc[0] if df[col].notnull().any() else None for col in df.columns],
    })
    all_dicts.append(d)

master_data_dict = pd.concat(all_dicts, ignore_index=True)
display(master_data_dict)

master_data_dict.to_csv('master_data_dictionary.csv', index=False)
print("\nSaved as 'master_data_dictionary.csv' — covers all 10 files, share with the team.")


## 10. Final Dataset Overview Summary

In [ ]:
# Cell 13: Final printed summary — copy into your report/presentation
print("="*70)
print("DATASET OVERVIEW SUMMARY — Disease Surveillance Project (All 10 Files)")
print("="*70)
for name, df in dfs.items():
    print(f"{name:28s} : {df.shape[0]:>7,} rows  x  {df.shape[1]:>2} cols   |  missing: {df.isnull().sum().sum():>5}  |  dupes: {df.duplicated().sum()}")

print("-"*70)
print(f"Total states in dim_state       : {dim_state.shape[0]}")
print(f"Total diseases in dim_disease   : {dim_disease.shape[0]}")
print(f"Total programs in dim_program   : {dim_program.shape[0]}")
print(f"Total sources in dim_source     : {dim_source.shape[0]}")
print(f"Total dates in dim_dates        : {dim_dates.shape[0]} (spanning {dim_dates_parsed['full_date'].min().date()} to {dim_dates_parsed['full_date'].max().date()})")
print("="*70)
